In [14]:
checkpoint = "microsoft/deberta-v3-base" 
tokenizer_checkpoint = "microsoft/deberta-v3-base"
dataset_name = "imdb"

In [12]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    print("Using CUDA device")
    print("Device name:", props.name)
    print(f"Compute capability: {props.major}.{props.minor}")
    print(f"Total GPU memory: {props.total_memory / 1024**3:.2f} GB")
else:
    DEVICE = torch.device("cpu")
    print("CUDA not available — using CPU")

print("DEVICE =", DEVICE)

Using CUDA device
Device name: NVIDIA GeForce RTX 4080
Compute capability: 8.9
Total GPU memory: 15.57 GB
DEVICE = cuda


In [8]:
from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

INFO     Tokenizing dataset imdb with AutoTokenizer for microsoft/deberta-v2-xlarge.
Map: 100%|██████████| 50000/50000 [00:08<00:00, 5705.15 examples/s]


In [17]:
from transformers import AutoModelForSequenceClassification
from chop import MaseGraph
import chop.passes as passes

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
model.config.problem_type = "single_label_classification"

import torch
from transformers.models.deberta_v2.modeling_deberta_v2 import make_log_bucket_position

# Tell the tracer: "If you see this function, don't go inside it. 
# Just record that we used it and move on."
torch.fx.wrap(make_log_bucket_position)

# NOW you can run your MaseGraph code
mg = MaseGraph(
    model,
    hf_input_names=["input_ids", "attention_mask", "labels"],
)

mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(mg)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AttributeError: 'torch._C.ScriptFunction' object has no attribute '__name__'

In [ ]:
from chop.passes.module import report_trainable_parameters_analysis_pass

_, _ = report_trainable_parameters_analysis_pass(mg.model)

# RoBERTa uses `model.roberta` instead of `model.bert`
for param in mg.model.roberta.embeddings.parameters():
    param.requires_grad = False

NameError: name 'mg' is not defined

In [ ]:
from chop.tools import get_trainer

trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)

eval_results = trainer.evaluate()
print(f"Pre-training accuracy: {eval_results['eval_accuracy']}")

trainer.train()

eval_results = trainer.evaluate()
print(f"Post-training accuracy: {eval_results['eval_accuracy']}")

/vol/bitbucket/ug22/adls-project/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Pre-training accuracy: 0.49764


Step,Training Loss
500,0.506900
1000,0.363800
1500,0.350800
2000,0.286700
2500,0.259600
3000,0.241800


Post-training accuracy: 0.9352


In [ ]:
mg.export("/vol/bitbucket/ug22/adls-data/models/modernbert-imdb-baseline")

INFO     Exporting MaseGraph to /vol/bitbucket/ug22/adls-data/models/roberta-imdb-baseline.pt, /vol/bitbucket/ug22/adls-data/models/roberta-imdb-baseline.mz
INFO     Exporting GraphModule to /vol/bitbucket/ug22/adls-data/models/roberta-imdb-baseline.pt
INFO     Saving full model format
INFO     Exporting MaseMetadata to /vol/bitbucket/ug22/adls-data/models/roberta-imdb-baseline.mz
WARNING  Failed to pickle call_function node: finfo
WARNING  cannot pickle 'torch.finfo' object
WARNING  Failed to pickle call_function node: getattr_2
WARNING  cannot pickle 'torch.finfo' object


In [ ]:
NotImplementedError: Model ModernBertForSequenceClassification is not supported yet, supported models: AlbertForMaskedLM, AlbertForMultipleChoice, AlbertForPreTraining, AlbertForQuestionAnswering, AlbertForSequenceClassification, AlbertForTokenClassification, AlbertModel, AltCLIPModel, AltCLIPTextModel, AltCLIPVisionModel, BartForCausalLM, BartForConditionalGeneration, BartForQuestionAnswering, BartForSequenceClassification, BartModel, BertForMaskedLM, BertForMultipleChoice, BertForNextSentencePrediction, BertForPreTraining, BertForQuestionAnswering, BertForSequenceClassification, BertForTokenClassification, BertLMHeadModel, BertModel, BlenderbotForCausalLM, BlenderbotForConditionalGeneration, BlenderbotModel, BlenderbotSmallForCausalLM, BlenderbotSmallForConditionalGeneration, BlenderbotSmallModel, BloomForCausalLM, BloomForQuestionAnswering, BloomForSequenceClassification, BloomForTokenClassification, BloomModel, CLIPForImageClassification, CLIPModel, CLIPTextModel, CLIPTextModelWithProjection, CLIPVisionModel, CLIPVisionModelWithProjection, CohereForCausalLM, CohereModel, ConvNextBackbone, ConvNextForImageClassification, ConvNextModel, DebertaForMaskedLM, DebertaForQuestionAnswering, DebertaForSequenceClassification, DebertaForTokenClassification, DebertaModel, DebertaV2ForMaskedLM, DebertaV2ForMultipleChoice, DebertaV2ForQuestionAnswering, DebertaV2ForSequenceClassification, DebertaV2ForTokenClassification, DebertaV2Model, Dinov2Backbone, Dinov2ForImageClassification, Dinov2Model, DistilBertForMaskedLM, DistilBertForMultipleChoice, DistilBertForQuestionAnswering, DistilBertForSequenceClassification, DistilBertForTokenClassification, DistilBertModel, DonutSwinModel, ElectraForCausalLM, ElectraForMaskedLM, ElectraForMultipleChoice, ElectraForPreTraining, ElectraForQuestionAnswering, ElectraForSequenceClassification, ElectraForTokenClassification, ElectraModel, GPT2DoubleHeadsModel, GPT2ForQuestionAnswering, GPT2ForSequenceClassification, GPT2ForTokenClassification, GPT2LMHeadModel, GPT2Model, GPTJForCausalLM, GPTJForQuestionAnswering, GPTJForSequenceClassification, GPTJModel, GPTNeoForCausalLM, GPTNeoForQuestionAnswering, GPTNeoForSequenceClassification, GPTNeoForTokenClassification, GPTNeoModel, GitVisionModel, HieraBackbone, HieraForImageClassification, HieraForPreTraining, HieraModel, HubertForCTC, HubertForSequenceClassification, HubertModel, IJepaForImageClassification, IJepaModel, LayoutLMForMaskedLM, LayoutLMForQuestionAnswering, LayoutLMForSequenceClassification, LayoutLMForTokenClassification, LayoutLMModel, LlamaForCausalLM, LlamaForQuestionAnswering, LlamaForSequenceClassification, LlamaForTokenClassification, LlamaModel, LxmertForPreTraining, LxmertForQuestionAnswering, LxmertModel, M2M100ForConditionalGeneration, M2M100Model, MBartForCausalLM, MBartForConditionalGeneration, MBartForQuestionAnswering, MBartForSequenceClassification, MBartModel, MT5ForConditionalGeneration, MT5ForQuestionAnswering, MT5ForSequenceClassification, MT5ForTokenClassification, MT5Model, MarianForCausalLM, MarianMTModel, MarianModel, MegatronBertForCausalLM, MegatronBertForMaskedLM, MegatronBertForMultipleChoice, MegatronBertForNextSentencePrediction, MegatronBertForPreTraining, MegatronBertForQuestionAnswering, MegatronBertForSequenceClassification, MegatronBertForTokenClassification, MegatronBertModel, MistralForCausalLM, MistralForQuestionAnswering, MistralForSequenceClassification, MistralForTokenClassification, MistralModel, MixtralForCausalLM, MixtralForQuestionAnswering, MixtralForSequenceClassification, MixtralForTokenClassification, MixtralModel, MobileBertForMaskedLM, MobileBertForMultipleChoice, MobileBertForNextSentencePrediction, MobileBertForPreTraining, MobileBertForQuestionAnswering, MobileBertForSequenceClassification, MobileBertForTokenClassification, MobileBertModel, NezhaForMaskedLM, NezhaForMultipleChoice, NezhaForNextSentencePrediction, NezhaForPreTraining, NezhaForQuestionAnswering, NezhaForSequenceClassification, NezhaForTokenClassification, NezhaModel, OPTForCausalLM, OPTForQuestionAnswering, OPTForSequenceClassification, OPTModel, PLBartForCausalLM, PLBartForConditionalGeneration, PLBartForSequenceClassification, PLBartModel, PeftModelForCausalLM, PeftModelForSeq2SeqLM, PegasusForCausalLM, PegasusForConditionalGeneration, PegasusModel, Qwen2ForCausalLM, Qwen2ForQuestionAnswering, Qwen2ForSequenceClassification, Qwen2ForTokenClassification, Qwen2Model, Qwen2MoeForCausalLM, Qwen2MoeForQuestionAnswering, Qwen2MoeForSequenceClassification, Qwen2MoeForTokenClassification, Qwen2MoeModel, Qwen3ForCausalLM, Qwen3ForQuestionAnswering, Qwen3ForSequenceClassification, Qwen3ForTokenClassification, Qwen3Model, Qwen3MoeForCausalLM, Qwen3MoeForQuestionAnswering, Qwen3MoeForSequenceClassification, Qwen3MoeForTokenClassification, Qwen3MoeModel, ResNetBackbone, ResNetForImageClassification, ResNetModel, RobertaForCausalLM, RobertaForMaskedLM, RobertaForMultipleChoice, RobertaForQuestionAnswering, RobertaForSequenceClassification, RobertaForTokenClassification, RobertaModel, SegformerForImageClassification, SegformerForSemanticSegmentation, SegformerModel, Speech2Text2Decoder, Speech2Text2ForCausalLM, Speech2TextForConditionalGeneration, Speech2TextModel, SwinBackbone, SwinForImageClassification, SwinForMaskedImageModeling, SwinModel, T5ForConditionalGeneration, T5ForQuestionAnswering, T5ForSequenceClassification, T5ForTokenClassification, T5Model, TrOCRDecoder, TrOCRForCausalLM, ViTForImageClassification, ViTForMaskedImageModeling, ViTModel, Wav2Vec2ForCTC, Wav2Vec2ForMaskedLM, Wav2Vec2ForPreTraining, Wav2Vec2ForSequenceClassification, Wav2Vec2Model, XGLMForCausalLM, XGLMModel
